In [10]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
# Cell 2 — Load raw data
df = pd.read_csv("../data/spy_raw.csv", header=[0,1], index_col=0, parse_dates=True)
df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
print(df.shape)

(2765, 5)


In [12]:
# Cell 3 — Feature engineering
df['Daily_Return'] = df['Close'].pct_change()
df['Close_Lag_1'] = df['Close'].shift(1)
df['Close_Lag_5'] = df['Close'].shift(5)
df['Close_Lag_20'] = df['Close'].shift(20)
df['Volatility_30'] = df['Daily_Return'].rolling(window=30).std()
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()
print(df.head(10))

                 Close        High         Low        Open     Volume  \
Date                                                                    
2015-01-02  170.125000  171.325815  169.089824  170.911744  121465900   
2015-01-05  167.052628  169.247196  166.746219  169.081571  169632600   
2015-01-06  165.479141  167.880745  164.684120  167.359012  209151400   
2015-01-07  167.541183  167.880724  166.356948  166.804139  125346700   
2015-01-08  170.514206  170.729531  168.932466  168.949020  147217800   
2015-01-09  169.147812  170.944876  168.534983  170.928310  158567300   
2015-01-12  167.822739  169.437623  167.218199  169.280275  144396100   
2015-01-13  167.350784  170.166459  166.050596  169.040187  214553300   
2015-01-14  166.340408  166.539167  164.443971  165.338352  192991100   
2015-01-15  164.816620  167.292753  164.700681  166.978068  176613900   

            Daily_Return  Close_Lag_1  Close_Lag_5  Close_Lag_20  \
Date                                                   

In [13]:
# Cell 4 — Define target variable (next day return, not price)
df['Target'] = df['Daily_Return'].shift(-1)
print("Target sample:")
print(df[['Close', 'Daily_Return', 'Target']].head())

Target sample:
                 Close  Daily_Return    Target
Date                                          
2015-01-02  170.125000           NaN -0.018059
2015-01-05  167.052628     -0.018059 -0.009419
2015-01-06  165.479141     -0.009419  0.012461
2015-01-07  167.541183      0.012461  0.017745
2015-01-08  170.514206      0.017745 -0.008013


In [14]:
# Cell 5 — Drop nulls created by lag and rolling features
print(f"Shape before dropping nulls: {df.shape}")
df = df.dropna()
print(f"Shape after dropping nulls: {df.shape}")

Shape before dropping nulls: (2765, 13)
Shape after dropping nulls: (2715, 13)


In [15]:
# Cell 6 — Train/test split (time-based, no shuffling)
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]
print(f"Train: {train.shape} | {train.index[0]} to {train.index[-1]}")
print(f"Test:  {test.shape} | {test.index[0]} to {test.index[-1]}")

Train: (2172, 13) | 2015-03-16 00:00:00 to 2023-10-27 00:00:00
Test:  (543, 13) | 2023-10-30 00:00:00 to 2025-12-29 00:00:00


In [16]:
# Cell 7 — Save processed data
df.to_csv("../data/spy_processed.csv")
train.to_csv("../data/spy_train.csv")
test.to_csv("../data/spy_test.csv")
print("Saved: spy_processed.csv, spy_train.csv, spy_test.csv")

Saved: spy_processed.csv, spy_train.csv, spy_test.csv
